# Vagary 原创填词作品统计

这是面向 Jupyter 的交互式统计本。请从上到下运行代码单元格（选中后按 `Shift + Enter`）。

> **首次使用**：请先在项目根目录执行 `pip install -e .` 安装 vagary 包，
> 然后点击工具栏的「全部运行」即可。

统计口径：一个词在同一首歌中出现多次，只按 **1 首作品** 计算。

In [ ]:
# ===== 首次使用才需要运行此单元格 =====
# 如果 vagary 包尚未安装，取消下面两行的注释并运行一次，完成后重启内核。
# import sys; sys.path.insert(0, '..')
# !pip install -e ..

# 载入本项目的函数
from pathlib import Path

from vagary import (
    load_works, word_frequency, words_by_length, plot_frequency,
    search_word, song_high_frequency_words, collaborator_frequency,
    plot_collaborators, export_tables,
)
import matplotlib.pyplot as plt

print('准备完成。')

## 1. 读取数据与设置排除词、个人词典

`EXCLUDE_WORDS` 里的词将完全不计入统计。个人词典是可选项：
将 `data/personal_dict_example.txt` 复制为 `data/personal_dict.txt` 后填写文件名。

In [ ]:
# 这里可以随时增删不希望被统计的词。
EXCLUDE_WORDS = {'我们', '你们','一个','什么','我的','你的','这一','的我','是我','的人'}

# 个人词典：若 data/personal_dict.txt 存在则使用，否则为 None
# 注意：本 Notebook 在 tests/ 目录下，数据文件在上级 data/ 目录
DATA_DIR = Path('..') / 'data'
USER_DICT = str(DATA_DIR / 'personal_dict.txt') if (DATA_DIR / 'personal_dict.txt').exists() else None

works = load_works(DATA_DIR / 'vagary.xlsx')
print(f'成功读取 {len(works)} 首作品。')
works[['歌曲名', '演唱', '作曲', '编曲']].head()

## 2. 词频统计与图表

默认优先用 jieba 分词。若本机没有 jieba，会自动以保底算法运行；
正式结果请安装依赖并可使用个人词典优化。

In [ ]:
frequency = word_frequency(works, exclude_words=EXCLUDE_WORDS, user_dict=USER_DICT)
print(f'共得到 {len(frequency)} 个候选词。')
# 分别查看 2 字、3 字、4 字词；此处显示各自前 20 名。
two_char = words_by_length(frequency, 2, top_n=20)
three_char = words_by_length(frequency, 3, top_n=20)
four_char = words_by_length(frequency, 4, top_n=20)
two_char

In [ ]:
# 将 length 改成 3 或 4，即可画对应字数的高频词图。
ax = plot_frequency(frequency, length=2, top_n=20, title='2 字高频词（按作品数）')
plt.show()
# 需要图片文件时，取消下面一行的注释：
# ax.figure.savefig('2字高频词.png', dpi=180, bbox_inches='tight')

## 3. 查询

下方是可直接运行的示例。把词语或歌名改成你想查的内容即可。
查词会显示包含该词的完整歌词。

In [ ]:
# 查询「夜色」在哪些作品中出现；结果中的「命中歌词」列为该作品中命中的歌词行。
search_word(works, '夜色', user_dict=USER_DICT)

In [ ]:
# 查询每首歌曲中，排名前 n 的高频词出现了几个，出现了哪些

import pandas as pd

# 临时设置最大显示行数为无限，显示完自动恢复默认
with pd.option_context('display.max_rows', None):
    display(song_high_frequency_words(works, frequency, n=30, user_dict=USER_DICT))

## 4. 合作者统计

三种口径均以作品数统计，同一人在一首歌中既作曲又编曲也只算一次。

In [ ]:
all_collabs = collaborator_frequency(works, '全部合作')
singer_collabs = collaborator_frequency(works, '演唱合作')
music_collabs = collaborator_frequency(works, '曲合作')
print('全部合作（前 20 名）')
display(all_collabs.head(20))
print('演唱合作（前 20 名）')
display(singer_collabs.head(20))
print('曲合作：作曲与编曲合并（前 20 名）')
display(music_collabs.head(20))

In [ ]:
# 把 all_collabs 换成 singer_collabs 或 music_collabs，可绘制另两种合作图。
ax = plot_collaborators(all_collabs, top_n=20, title='全部合作统计（前 20 名）')
plt.show()

## 5. 导出统计表

运行后会在项目根目录的 `output/` 文件夹中生成 Excel 统计结果，
包含全部词频、2/3/4 字词频、三种合作统计。

In [ ]:
# 输出到项目根目录的 output/ 文件夹
OUTPUT_DIR = Path('..') / 'output'
output_file = export_tables(OUTPUT_DIR, frequency, works)
print(f'已导出：{output_file.resolve()}')